<a href="https://colab.research.google.com/github/hoofs-lab/Prueba_Peloton/blob/main/SEC_Hoof.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q edgartools pandas duckdb

In [29]:
import duckdb
import pandas as pd
from edgar import *

Filter by Form Type

In [32]:
from edgar import set_identity, Company

# 1. SEC Identity Requirement Setup
print("Setting SEC User-Agent Identity...")
name = input("Enter First Name: ").strip()
last_name = input("Enter Last Name: ").strip()
email = input("Enter Email: ").strip()
identity_string = f"{name} {last_name} {email}"
set_identity(identity_string)
print(f"SEC User-Agent Identity set to: '{identity_string}'\n")

# Outer loop to allow adding/searching other companies sequentially
while True:
    # 2. Select Company by Ticker or CIK
    while True:
        ticker_or_cik = input("Enter Ticker or CIK (e.g., PTON or 0001826209): ").strip()
        try:
            company = Company(ticker_or_cik)
            print(f"Successfully loaded company: {company}\n")
            break
        except Exception as e:
            print(f"Error: Could not find company with '{ticker_or_cik}'. Please try again.\n")

    # 3. Interactive Menu to select multiple form types
    print("Select the Form Type(s) to retrieve (separate multiple choices with commas, e.g., 1,2):")
    print("1. 10-K (Annual Reports)")
    print("2. 10-Q (Quarterly Reports)")
    print("3. 8-K (Current Reports)")

    selected_forms = []
    while True:
        choice = input("Enter choices (e.g., 1 or 1,2 or 1,2,3): ").strip()
        parts = [p.strip() for p in choice.split(",")]

        valid = True
        temp_forms = []
        for part in parts:
            if part == '1':
                temp_forms.append("10-K")
            elif part == '2':
                temp_forms.append("10-Q")
            elif part == '3':
                temp_forms.append("8-K")
            else:
                valid = False
                break

        if valid and temp_forms:
            selected_forms = temp_forms
            break
        else:
            print("Invalid input. Please enter numbers 1, 2, or 3 separated by commas.")

    # 4. Fetch and display length of selected filings
    for form_type in selected_forms:
        try:
            filings = company.get_filings(form=form_type)
            print(f"Total {form_type} filings for {company.name}: {len(filings)}")
        except Exception as e:
            print(f"Could not fetch {form_type} filings: {e}")

    # 5. Ask if the user wants to add/query another company
    another = input("\nWould you like to search another company? (y/n): ").strip().lower()
    if another != 'y':
        print("Exiting company query tool.")
        break

Setting SEC User-Agent Identity...
Enter First Name: Carlos
Enter Last Name: Guevara
Enter Email: hoofsdata@gmail.com
SEC User-Agent Identity set to: 'Carlos Guevara hoofsdata@gmail.com'

Enter Ticker or CIK (e.g., PTON or 0001826209): PTON
Successfully loaded company: Company(CIK=1639825)

Select the Form Type(s) to retrieve (separate multiple choices with commas, e.g., 1,2):
1. 10-K (Annual Reports)
2. 10-Q (Quarterly Reports)
3. 8-K (Current Reports)
Enter choices (e.g., 1 or 1,2 or 1,2,3): 1,2
Total 10-K filings for PELOTON INTERACTIVE, INC.: 7
Total 10-Q filings for PELOTON INTERACTIVE, INC.: 22

Would you like to search another company? (y/n): y
Enter Ticker or CIK (e.g., PTON or 0001826209): aapl
Successfully loaded company: Company(CIK=320193)

Select the Form Type(s) to retrieve (separate multiple choices with commas, e.g., 1,2):
1. 10-K (Annual Reports)
2. 10-Q (Quarterly Reports)
3. 8-K (Current Reports)
Enter choices (e.g., 1 or 1,2 or 1,2,3): 1,2
Total 10-K filings for App

##Set Range

In [34]:
# Prompt the user to enter a year
while True:
    year = input("Enter a year (e.g., 2024) or range (e.g., 2020-2024): ").strip()

    # Year or range
    if len(year) == 4 and year.isdigit():
        date_range = f"{year}-01-01:{year}-12-31"
        print(f"Year: {year}")
        break

    # Two years without hyphen: 8 digits (e.g., 20222023)
    elif len(year) == 8 and year.isdigit():
        start_year = year[:4]
        end_year = year[4:]
        date_range = f"{start_year}-01-01:{end_year}-12-31"
        print(f"Range: {date_range}")
        break

    # Two years with hyphen: 9 characters (e.g., 2022-2023)
    elif len(year) == 9 and year == "-" and year[:4].isdigit() and year[5:].isdigit():
        start_year, end_year = year.split("-")
        date_range = f"{start_year}-01-01:{end_year}-12-31"
        print(f"Range: {date_range}")
        break

    # Incorrect
    else:
        print("Please enter a correct year\n")

Enter a year (e.g., 2024) or range (e.g., 2020-2024): 2022
Year: 2022


In [33]:
from edgar import Company

# Ensure company object is loaded from the kernel state ticker
try:
    company_identifier = ticker_or_cik
except NameError:
    company_identifier = "PTON" # fallback default

company = Company(company_identifier)

# Filter by form type AND the chosen year
all_filings = company.get_filings(filing_date=date_range)
annual_reports = company.get_filings(form="10-K", filing_date=date_range)
quarterly_reports = company.get_filings(form="10-Q", filing_date=date_range)
current_reports = company.get_filings(form="8-K", filing_date=date_range)

print(f"\n--- Results for {year} ---")
print(f"Total filings:          {len(all_filings)}")
print(f"10-K annual reports:    {len(annual_reports)}")
print(f

_IncompleteInputError: incomplete input (2105833251.py, line 20)